## This code is meant to be used only once!!
The code generates functions from a neural network with a certain architecture. Make sure to backup the data generated with this code (functions and weights)

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import pickle
import random
from pathlib import Path
import os

random.seed(a=1111)

### Choose: 

1. Network architecture (input layer size, number of layers and layer size) 


2. Number of functions


3. Maximum value of x for functions 


4. Data resolutions 


5. Activation functions 

In [9]:
# Network Architecture
ILS = 1 #Input layer size
NL, LS = 10, 15 # Number of hidden layers and their size

NREP = 10 #Number of functions you want to generate
xmax=4;#step=0.1 #limits of x and resolution step to generate functions

steps=[0.1, 0.05, 0.025, 0.004]
activation_functions=['tanh', 'leaky_ReLU']

# Initialize weights
def init_w(ils=ILS, nl=NL, ls=LS):
    nu = [ils] + [ls] * nl
    w = []
    for l in range(nl):
        w.append([])
        for i in range(nu[l]):
            w[l].append([])
            for j in range(nu[l+1]):
                w[l][i].append(2*(np.random.uniform()-.5)**1)
    w.append([[(2*(np.random.uniform()-.5))**1 for i in range(nu[nl])]])
        
    return w

# Import weights
def import_w(file):
    with open(file, 'rb') as f:
        w = pickle.load(f)
    return w
    
# Compute output
def compute_output(invals, w, activation_fun):
    # Input layer
    cl = np.array(invals)
    # Hidden layers
    for l in range(len(w)-1):
        cl = np.array([np.sum([w[l][i][j]*cl[i] for i in range(len(cl))]) for j in range(len(w[l][0]))])
        if activation_fun=='tanh':
            cl = np.tanh(cl)
        elif activation_fun=='ReLU':
            cl=[cl_i if cl_i>0 else 0 for cl_i in cl]
        elif activation_fun=='leaky_ReLU':
            cl=[cl_i if cl_i>0 else 0.01*cl_i for cl_i in cl]
            
        
    return(np.sum([w[-1][i][0] * cl[i] for i in range(len(w[-1]))]))


#add Gaussian noise to data
def add_noise_to_data(activation_function, resolution_var, sigma_v, realizations):
    
    #read data                                                                                       
    input_path='../../data/alternative_experiments/ILS%d_NL%d_LS%d/' % (ILS, NL, LS)
    filename='NN_%s_NREP_%s_ILS%d_NL%d_LS%d_step_%s.csv'  %(str(function) , str(NREP), ILS, NL,  LS, str(step))
    data=input_path + filename

    output_path=input_path + 'noisy_data/' 

    try:
        os.makedirs(output_path)
    except FileExistsError:
                # directory already exists
        pass

    d0=pd.read_csv(data)
    d0=d0.drop(columns='Unnamed: 0')
    display(d0)
    display(d0.index)
    #add noise                                                                                       
    mean=0;sample=d0.index.stop

    for sigma in sigma_v:
        for r in range(realizations):

            noise = np.random.normal(mean,sigma,sample)

            #Add Gaussian noise to data                                                              
            d0['noise']=noise
            d0['y_noise']= d0['y'] + d0['noise']

            #Save data
            step_sub_path=output_path + '%s/' % step 

            try:
                os.makedirs(step_sub_path)
            except FileExistsError:
                # directory already exists
                pass
                
            #output_path='../../data/alternative_data/' + resolution_var + '_resolution/'
            #d_all.to_csv(output_path + 'NN_%s_NREP_%s_ILS%d_NL%d_LS%d_step_%s.csv'  %(str(function) , str(NREP), ILS, NL,  LS, str(step)))
            
            d0.to_csv(step_sub_path + 'NN_%s_ILS%d_NL%d_LS%d_sigma_%s_r_%s_step_%s.csv' \
                      %(str(function), ILS, NL,  LS, str(sigma), str(r), str(step)) )

    return None

The following cell creates the files with the dataframes in a specific subfolder of `data/`

In [3]:
#create new folder for files
#-------------------------------------------------------------------------------------
output_path='../../data/alternative_experiments/' + 'ILS%d_NL%d_LS%d/' % (ILS, NL, LS)

try:
    os.makedirs(output_path)
except FileExistsError:
    # directory already exists
    pass

#display(output_path)
#-------------------------------------------------------------------------------------

#create and save dataframes
#-------------------------------------------------------------------------------------
for function in activation_functions:
    weights=[]
    for step in steps:
        d_all = pd.DataFrame({'x1' : [], 'y': [], 'rep': []}) #initialize dataframe for normalized functions
        d_all_ie = pd.DataFrame({'x1' : [], 'y': [], 'rep': []}) #initialize dataframe for normalized Inter/Extrapolation functions
        d_all_raw = pd.DataFrame({'x1' : [], 'y_raw': [], 'rep': []}) #initialize dataframe for non-normalized functions
        
        for rep in range(NREP):
            
            w = init_w(ils=ILS+1, nl=NL, ls=LS)
            weights.append(w)

            #approximation datasets
            #--------------------------------------------------------------
            #initialize x and generate y values
            x1 = np.arange(-xmax, xmax, step)
            x1s, ys = [], []
            y = [compute_output([1, thisx1], w, function) for thisx1 in x1]
            x1s=x1
            ys=y
    
            # Normalize functions
            ys = np.array(ys)
            ys_norm = (ys - min(ys)) / (max(ys) - min(ys) + 1e-15)

            #Save normalized functions to dataframes
            d = pd.DataFrame({'x1' : x1s, 'y' : ys_norm, 'rep': rep})
            d_all=pd.concat([d_all,d])
    
            #Save non-normalized functions
            d_raw = pd.DataFrame({'x1' : x1s, 'y_raw' : ys, 'rep': rep})
            d_all_raw=pd.concat([d_all_raw,d_raw])
            #--------------------------------------------------------------

            #interpolation/extrapolation datasets
            #--------------------------------------------------------------
            inter_extrap_step=step/5
            
            #initialize x and generate y values
            x_ie_1 = np.arange(-xmax, xmax, inter_extrap_step)
            x_ie_1s, y_ie_s = [], []
            y_ie = [compute_output([1, thisx_ie_1], w, function) for thisx_ie_1 in x_ie_1]
            x_ie_1s=x_ie_1
            y_ie_s=y_ie
    
            # Normalize functions
            y_ie_s = np.array(y_ie_s)
            y_ie_s_norm = (y_ie_s - min(ys)) / (max(ys) - min(ys) + 1e-15) #same normalization as "normal" dataframes

            #Save normalized functions to dataframes
            d_ie = pd.DataFrame({'x1' : x_ie_1s, 'y' : y_ie_s_norm, 'rep': rep})
            d_all_ie=pd.concat([d_all_ie,d_ie])
            #--------------------------------------------------------------
            
            
        if step==0.05:
            with open (output_path + 'NN_weights_%s_NREP_%s_ILS%d_NL%d_LS%d.pickle' %(str(function) , str(NREP), ILS, NL, LS), 'wb') as f:
                pickle.dump(weights, f, protocol=None)

        d_all.to_csv(output_path + 'NN_%s_NREP_%s_ILS%d_NL%d_LS%d_step_%s.csv'  %(str(function) , str(NREP), ILS, NL,  LS, str(step)))
        d_all_raw.to_csv(output_path + 'non_normalized_NN_%s_NREP_%s_ILS%d_NL%d_LS%d_step_%s.csv' %(str(function), str(NREP), ILS, NL,  LS, str(step)))

        
        #display(inter_extrap_step)
        d_all_ie.to_csv(output_path + 'NN_%s_NREP_%s_ILS%d_NL%d_LS%d_step_%s_interpolation_data.csv'  %\
                     (str(function) , str(NREP), ILS, NL,  LS, str(inter_extrap_step)))
#-------------------------------------------------------------------------------------

Choose the maximum level of noise, the discrete steps of increasing noise and number of noise realizations per level of noise
The resulting dataframes are saved in a specific subfolder

In [8]:
#Add noise to approximation data
sigma_max=0.2 #maximum level of Gaussian noise
sigma_step=0.01 #Gaussian noise steps
r=3 #number of noise realizations per step

sigmas=[i for i in np.arange(0,sigma_max + sigma_step,sigma_step)]

for function in activation_functions:
    for step in steps:
        print(function)
        print(step)
        add_noise_to_data(function, step, sigmas, r)






tanh
0.1


,x1,y,rep
0,-4.0,0.999970,0.0
1,-3.9,1.000000,0.0
2,-3.8,0.999919,0.0
3,-3.7,0.999701,0.0
4,-3.6,0.999319,0.0
...,...,...,...
795,3.5,0.773641,9.0
796,3.6,0.756928,9.0
797,3.7,0.745699,9.0
798,3.8,0.737627,9.0


RangeIndex(start=0, stop=800, step=1)

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/'

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/0.1/'

...

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/0.004/'

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/'

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/0.004/'

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/'

'../../data/alternative_experiments/ILS1_NL10_LS15/noisy_data/0.004/'

In [ ]:
#PLOT -2,2

#Figure Size                                                                                                                                                                                                
cm = 1/2.54  # centimeters in inches                                                                                                                                                                        
width=45*cm;height=14*cm #Width and height of plots 
matplotlib.rcParams['figure.figsize'] = [width, height]

rows=2;cols=5
gs=gridspec.GridSpec(rows,cols)
gs.update(left=0.1,right=0.99,bottom=0.08,top=0.97,wspace=0.25,hspace=0.0)

#Plot train rank (-2,2)
h=0
for r in range(rows):
    for c in range(cols):
        ax_rc=plt.subplot(gs[r,c])
        d=d_all[d_all['rep']==h]
        ax_rc=sns.lineplot(data=d, x='x1', y='y')
        ax_rc.set_xlim(-2,2)
        h+=1
#plt.savefig('../../results/seminal_data/' + 'NN_function_' + activation_function + '_' + 'ILS%d_NL%d_LS%d'  %(ILS, NL,  LS) + '.png', dpi=300)

In [ ]:
#PLOT FULL RANK (-4,4)

#Figure Size                                                                                                                                                                                                
cm = 1/2.54  # centimeters in inches                                                                                                                                                                        
width=45*cm;height=14*cm #Width and height of plots 
matplotlib.rcParams['figure.figsize'] = [width, height]

h=0
for r in range(rows):
    for c in range(cols):
        ax_rc=plt.subplot(gs[r,c])
        d=d_all[d_all['rep']==h]
        ax_rc=sns.lineplot(data=d, x='x1', y='y')
        ax_rc.vlines(x=[-2, 2], ymin=0, ymax=1, color='k',linestyle='--')
        ax_rc.set_xlim(-4,4)
        h+=1
#plt.savefig('../../results/seminal_data/' + 'NN_function_full_rank_' + activation_function + '_' + 'ILS%d_NL%d_LS%d'  %(ILS, NL,  LS) + '.png', dpi=300)